# Exploring a Prepared Table

*Inspect numeric distributions and category frequencies before drawing conclusions.*


## Before You Run

In **Add files**, select `retail_sales_clean.csv` from this lesson's supplied assets.
Enter `assets/07_exploratory_analysis` for **Save folder**. The upload creates
missing folders and places the file at
`/home/pyodide/assets/07_exploratory_analysis/retail_sales_clean.csv`.

`/home/pyodide` is the browser's virtual filesystem, not a folder on your
computer. The upload folder is relative to `/home/pyodide`; Python relative
paths use the current directory returned by `os.getcwd()`. Keep the default
working directory in the course site. Locally, run from this chapter folder.
Uploading again to the same complete path replaces that input file.
Saved examples appear under `outputs/07_exploratory_analysis/` in the file browser.


## Inspect the Prepared Dataset

This supplied 15-order file is a separate exploration example from the
reference material. It is not the exact result of cleaning the 12-row raw
file: it has additional orders and uses quantity 4 for order 1005.
Check its shape, fields, and missing values before interpreting summaries.


In [1]:
import os

import pandas as pd

input_directory = os.path.join("assets", "07_exploratory_analysis")
output_directory = os.path.join("outputs", "07_exploratory_analysis")
input_path = os.path.join(input_directory, "retail_sales_clean.csv")
output_path = os.path.join(output_directory, "category_counts.csv")
os.makedirs(output_directory, exist_ok=True)

sales_df = pd.read_csv(input_path, encoding="utf-8")
print("Prepared sales preview:")
print(sales_df.head(3).to_string(index=False))
print("Shape:", sales_df.shape)
print("Columns:", sales_df.columns.tolist())
print("Missing values:")
print(sales_df.isna().sum())


Prepared sales preview:
 order_id order_date region  category      product customer_type  quantity  unit_price
     1001 2026-01-03  Seoul    Laptop Notebook Pro     Corporate         2        1200
     1002 2026-01-05  Busan Accessory        Mouse    Individual         5          25
     1003 2026-01-07  Seoul   Monitor UltraView 27     Corporate         1         310
Shape: (15, 8)
Columns: ['order_id', 'order_date', 'region', 'category', 'product', 'customer_type', 'quantity', 'unit_price']
Missing values:
order_id         0
order_date       0
region           0
category         0
product          0
customer_type    0
quantity         0
unit_price       0
dtype: int64


The table has 15 rows and 8 columns with no missing cells. It uses `product` as its product-name field and has no channel column.


## Describe Numeric Values

`.describe()` reports non-missing count, mean, sample standard deviation,
minimum, quartiles, and maximum for the selected numeric columns.
The `50%` row is the median. The `std` row uses `ddof=1`, unlike
NumPy's default `ddof=0` standard deviation.


In [2]:
print("Numeric summary:")
print(sales_df[["quantity", "unit_price"]].describe().round(2))


Numeric summary:
       quantity  unit_price
count     15.00       15.00
mean       2.60      462.33
std        2.38      490.48
min        1.00       25.00
25%        1.00       45.00
50%        2.00      310.00
75%        3.00      980.00
max       10.00     1200.00


The median quantity is 2 and the maximum is 10, so 10 is well above the middle value. The mean quantity is 2.6. This small summary suggests inspecting large orders; it does not by itself prove an error.


## Count Categories and Measure Their Shares

`.value_counts()` counts records for each distinct value.
`normalize=True` divides by the number of non-missing values to return
proportions. These are shares of orders, not shares of revenue.
`.sort_index()` gives the display a predictable alphabetical order.


In [3]:
category_counts = sales_df["category"].value_counts().sort_index()
customer_shares = sales_df["customer_type"].value_counts(normalize=True).sort_index()

print("Order counts by category:")
print(category_counts)
print("Customer type shares:")
print(customer_shares.round(4))
print("Order counts accounted for:", category_counts.sum())


Order counts by category:
category
Accessory    7
Laptop       5
Monitor      3
Name: count, dtype: int64
Customer type shares:
customer_type
Corporate     0.5333
Individual    0.4667
Name: proportion, dtype: float64
Order counts accounted for: 15


Accessory, Laptop, and Monitor have 7, 5, and 3 orders. Corporate customers account for 8/15 (about 53.33%) and Individual customers for 7/15 (about 46.67%).


## Save a Category Report

`.rename_axis("category")` names the Series index; `.reset_index(name=...)`
converts its labels and values into two ordinary DataFrame columns. This
makes both fields explicit when saving with `index=False`.


In [4]:
category_report = category_counts.rename_axis("category").reset_index(name="order_count")
category_report.to_csv(output_path, index=False, encoding="utf-8")
reopened_df = pd.read_csv(output_path, encoding="utf-8")

print("Saved file:", output_path)
print("Reopened category report:")
print(reopened_df)
print("Round trip matches:", reopened_df.equals(category_report))


Saved file: outputs/07_exploratory_analysis/category_counts.csv
Reopened category report:
    category  order_count
0  Accessory            7
1     Laptop            5
2    Monitor            3
Round trip matches: True


The report retains three categories and a total count of 15. The reopened table equals the expected report.


Explore both numeric distributions and category frequencies. State whether a proportion counts records, customers, quantities, or revenue.

[Descriptive summaries](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)
